# Neural Network Lab — Notebook thí nghiệm

Notebook này thực hiện các mục Part 0–4; các con số trong ô kết quả đến từ lần chạy thực tế. Dự đoán, nhận xét và câu trả lời trong bài được viết bằng tiếng Việt.

Quy trình cuối gồm khởi động lại kernel, chạy lần lượt toàn bộ notebook, lưu output và kiểm tra các tệp trong thư mục nộp.


In [1]:
# ===== Cấu hình đường dẫn và môi trường =====
import os, sys, json, time, subprocess, math, statistics
from pathlib import Path
import numpy as np, torch
import matplotlib.pyplot as plt

# Hỗ trợ chạy từ thư mục repo hoặc từ submission_2A202602793/code/.
_repo_candidate = Path("../..").resolve()
if not (_repo_candidate / "scripts" / "split_data.py").is_file():
    _repo_candidate = Path.cwd().resolve()
if not (_repo_candidate / "scripts" / "split_data.py").is_file():
    raise FileNotFoundError("Không tìm thấy repo root có scripts/split_data.py; hãy đặt REPO_ROOT đúng.")
REPO_ROOT = str(_repo_candidate)
OUT_DIR = str(Path(REPO_ROOT) / "submission_2A202602793")
CODE_DIR = str(Path(OUT_DIR) / "code")
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)
# Colab: clone repo rồi đặt REPO_ROOT = "/content/K4-Track4-Day1-NeuralNetwork".
# !git clone https://github.com/VinUni-AI20k/K4-Track4-Day1-NeuralNetwork.git

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Phiên bản PyTorch: {torch.__version__}")
print(f"Thiết bị: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
os.makedirs(f"{OUT_DIR}/figures", exist_ok=True)
os.makedirs(f"{OUT_DIR}/results", exist_ok=True)

from data import prepare_data, load_split, make_val_split, iterate_batches
from model import MLP, EXPECTED_PARAMS, count_params, init_weights, activation_stats
from optimizer import build_optimizer, clip_gradients
from train import DEFAULT_CFG, set_seed, evaluate, predict, run_experiment, final_eval
from plots import plot_run, plot_compare
from results_table import save_result, load_results, to_row, write_xlsx


Phiên bản PyTorch: 2.6.0
Thiết bị: cpu


## Part 0 — Dữ liệu
Chia sẵn bằng metadata: chạy `scripts/split_data.py` từ thư mục gốc repo (tạo `data/processed/train.npz`, `eval.npz`).
Sau đó tách validation từ train, chuẩn hoá bằng thống kê của train, đưa lên device.

In [2]:
split_process = subprocess.run(
    [sys.executable, "scripts/split_data.py"], cwd=REPO_ROOT, check=True,
    text=True, capture_output=True
)
print(split_process.stdout)
data = prepare_data(device, val_fraction=0.2, seed=42, processed_dir=f"{REPO_ROOT}/data/processed")

X_train_full, y_train_full, X_eval_raw, y_eval_raw, row_ids_raw = load_split(f"{REPO_ROOT}/data/processed")
X_tr_raw, y_tr_raw, X_val_raw, y_val_raw = make_val_split(X_train_full, y_train_full, val_fraction=0.2, seed=42)
for split_name, features, labels in (
    ("train", data["X_tr"], data["y_tr"]),
    ("val", data["X_val"], data["y_val"]),
    ("eval", data["X_eval"], data["y_eval"]),
):
    print(f"{split_name}: X={tuple(features.shape)} {features.dtype}, y={tuple(labels.shape)} {labels.dtype}, nhãn={int(labels.min())}..{int(labels.max())}")
print(f"eval_row_id: {data['eval_row_id'].shape}, dtype={data['eval_row_id'].dtype}")

train_numeric = data["X_tr"][:, :10].detach().cpu().numpy().astype(np.float64)
numeric_mean = train_numeric.mean(axis=0)
numeric_std = train_numeric.std(axis=0)
print("Trung bình 10 cột số trên X_tr:", np.round(numeric_mean, 7))
print("Độ lệch chuẩn 10 cột số trên X_tr:", np.round(numeric_std, 7))
assert np.allclose(numeric_mean, 0.0, atol=1e-5)
assert np.allclose(numeric_std, 1.0, atol=1e-4)
assert np.array_equal(data["X_tr"][:, 10:].detach().cpu().numpy(), X_tr_raw[:, 10:])
assert tuple(data["X_tr"].shape) == (371847, 54)
assert tuple(data["X_val"].shape) == (92962, 54)
assert tuple(data["X_eval"].shape) == (116203, 54)
assert len(data["eval_row_id"]) == 116203 and len(np.unique(data["eval_row_id"])) == 116203

majority_class = int(np.bincount(y_tr_raw, minlength=7).argmax())
majority_val_acc = float(np.mean(y_val_raw == majority_class))

_small_X = torch.zeros((1000, 54), dtype=torch.float32, device=device)
_small_y = torch.zeros(1000, dtype=torch.int64, device=device)
_batch_sizes = [len(yb) for _, yb in iterate_batches(_small_X, _small_y, 512, shuffle=False)]
print(f"Kích thước batch minh hoạ (N=1000, batch=512): {_batch_sizes}")
assert _batch_sizes == [512, 488]

train: X (464809, 54) float32, y (464809,) int64, nhãn 0..6 -> data/processed/train.npz
eval : X (116203, 54) float32, y (116203,) int64, nhãn 0..6 -> data/processed/eval.npz

lớp   train(%)   eval(%)
  0    36.461    36.460
  1    48.760    48.760
  2     6.154     6.154
  3     0.473     0.472
  4     1.634     1.634
  5     2.989     2.989
  6     3.530     3.530

đoán luôn lớp đa số (lớp 1) cho accuracy trên eval = 0.4876



X_tr=(371847, 54), X_val=(92962, 54), X_eval=(116203, 54)
val accuracy (luôn đoán lớp đa số train 1) = 0.4876


train: X=(371847, 54) torch.float32, y=(371847,) torch.int64, nhãn=0..6
val: X=(92962, 54) torch.float32, y=(92962,) torch.int64, nhãn=0..6
eval: X=(116203, 54) torch.float32, y=(116203,) torch.int64, nhãn=0..6
eval_row_id: (116203,), dtype=int64
Trung bình 10 cột số trên X_tr: [-2.e-07 -0.e+00  1.e-07  0.e+00  0.e+00  0.e+00 -2.e-07 -4.e-07 -1.e-07
  0.e+00]
Độ lệch chuẩn 10 cột số trên X_tr: [1.0000001 1.        1.        1.        1.        1.        1.
 1.        1.        1.       ]
Kích thước batch minh hoạ (N=1000, batch=512): [512, 488]


## Part 1 — Model và kiểm tra "sức khoẻ" ban đầu
Quy định: `M-base` = `54 → 256 → 128 → 7`, **47 879 tham số**, logits `(B, 7)`, không softmax trong model.

In [3]:
set_seed(42)
model = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
assert count_params(model) == EXPECTED_PARAMS[(256, 128)] == 47879
for hidden, expected in EXPECTED_PARAMS.items():
    check_model = MLP(hidden=hidden, dropout=0.0, init="he").to(device)
    print(f"hidden={hidden}: {count_params(check_model):,} tham số (kỳ vọng {expected:,})")
    assert count_params(check_model) == expected

probe = torch.randn(8, 54, device=device)
h = probe
for layer in model.net:
    h = layer(h)
    if isinstance(layer, torch.nn.Linear):
        print(f"Sau Linear {layer.in_features}→{layer.out_features}: {tuple(h.shape)}")
logits = h
assert logits.shape == (8, 7)
print("Shape logits:", tuple(logits.shape), "(logit thô, không softmax)")

step0_metrics = evaluate(model, data["X_val"], data["y_val"], loss_name="ce")
step0_loss = step0_metrics["loss"]
print(f"Loss CE bước 0 trên val: {step0_loss:.6f}; ln(7) = {math.log(7):.6f}")

# Quá khớp một lô cố định 20 mẫu train; đây không dùng val/eval để cập nhật.
overfit_x = data["X_tr"][:20]
overfit_y = data["y_tr"][:20]
overfit_model = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
overfit_optimizer = torch.optim.Adam(overfit_model.parameters(), lr=0.01)
overfit_steps, overfit_losses, overfit_accs = [0], [], []
with torch.no_grad():
    start_logits = overfit_model(overfit_x)
    start_loss = torch.nn.functional.cross_entropy(start_logits, overfit_y).item()
    start_acc = (start_logits.argmax(1) == overfit_y).float().mean().item()
overfit_losses.append(start_loss)
overfit_accs.append(start_acc)
for step in range(1, 1001):
    overfit_model.train()
    overfit_optimizer.zero_grad(set_to_none=True)
    overfit_loss = torch.nn.functional.cross_entropy(overfit_model(overfit_x), overfit_y)
    overfit_loss.backward()
    overfit_optimizer.step()
    if step % 10 == 0 or step == 1:
        overfit_model.eval()
        with torch.no_grad():
            check_logits = overfit_model(overfit_x)
            current_loss = float(torch.nn.functional.cross_entropy(check_logits, overfit_y).item())
            current_acc = float((check_logits.argmax(1) == overfit_y).float().mean().item())
        overfit_steps.append(step)
        overfit_losses.append(current_loss)
        overfit_accs.append(current_acc)
        if current_acc == 1.0 and current_loss < 1e-3:
            break
overfit_final_loss = overfit_losses[-1]
overfit_final_acc = overfit_accs[-1]
print(f"Quá khớp 20 mẫu: bước={overfit_steps[-1]}, loss={overfit_final_loss:.8f}, accuracy={overfit_final_acc:.1%}")
assert overfit_final_acc == 1.0 and overfit_final_loss < 1e-2
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(overfit_steps, overfit_losses, label="CE trên 20 mẫu")
ax.set(xlabel="Bước cập nhật", ylabel="Cross-entropy", title="Quá khớp một lô 20 mẫu")
ax.legend()
fig.tight_layout()
fig.savefig(f"{OUT_DIR}/figures/part1_overfit20.png", dpi=160, bbox_inches="tight")
plt.close(fig)

# Gradient baseline sau backward, trước optimizer.step().
model.train()
model.zero_grad(set_to_none=True)
grad_loss = torch.nn.functional.cross_entropy(model(data["X_tr"][:128]), data["y_tr"][:128])
grad_loss.backward()
gradient_norms = {}
for name, parameter in model.named_parameters():
    norm = None if parameter.grad is None else float(parameter.grad.detach().norm().item())
    gradient_norms[name] = norm
    print(f"{name}: grad norm = {norm}")
assert all(norm is not None and norm > 0 for norm in gradient_norms.values())
print("Kiểm tra Part 1 đạt: architecture, logits, loss bước 0, overfit và mọi gradient khác 0.")

hidden=(256, 128): 47,879 tham số (kỳ vọng 47,879)
hidden=(512, 256): 161,287 tham số (kỳ vọng 161,287)
hidden=(256, 128, 64): 55,687 tham số (kỳ vọng 55,687)
Sau Linear 54→256: (8, 256)
Sau Linear 256→128: (8, 128)
Sau Linear 128→7: (8, 7)
Shape logits: (8, 7) (logit thô, không softmax)
Loss CE bước 0 trên val: 2.377572; ln(7) = 1.945910


Quá khớp 20 mẫu: bước=30, loss=0.00030214, accuracy=100.0%
net.0.weight: grad norm = 0.641794741153717
net.0.bias: grad norm = 0.3837469220161438
net.2.weight: grad norm = 2.4385881423950195
net.2.bias: grad norm = 0.4997515082359314
net.4.weight: grad norm = 2.1445913314819336
net.4.bias: grad norm = 0.6131887435913086
Kiểm tra Part 1 đạt: architecture, logits, loss bước 0, overfit và mọi gradient khác 0.


**Nhận xét Part 1:** M-base có đúng 47.879 tham số và xuất logits `(8, 7)`. Loss CE bước 0 đo được là **2,3776**, cao hơn `ln(7)=1,9459` khoảng 0,4317. `ln(7)` là mốc khi logits bằng nhau; với khởi tạo He ngẫu nhiên, logits giữa các lớp chưa bằng nhau nên loss ban đầu có thể lệch lên. Shape, dữ liệu chuẩn hoá và nhãn đều đã qua kiểm tra. Mạng ghi nhớ 20 mẫu train sau 30 bước (loss 0,000302; accuracy 100%). Cả sáu tensor tham số đều có chuẩn gradient khác 0 sau `backward()`.

## Part 2 — Pipeline và baseline
`run_experiment(cfg, data)` nhận một dict cấu hình và trả về lịch sử + tóm tắt. Baseline: SGD+momentum 0.9, CE, He init, batch 512, 20 epoch.

In [4]:
# Tìm learning rate bằng validation; các lượt dò nhanh dùng cùng split và seed.
run_tag = time.strftime("%Y%m%d-%H%M%S")
lr_grid = [0.01, 0.03, 0.1]
lr_search_results = []
for lr in lr_grid:
    lr_label = f"{lr:g}".replace(".", "p")
    exp_id = f"{run_tag}-lr-sgdm-{lr_label}"
    lr_cfg = {**DEFAULT_CFG, "exp_id": exp_id, "group": "hparam",
               "description": f"Dò lr baseline nhanh, lr={lr:g}",
               "lr": lr, "epochs": 5, "seed": 42}
    lr_result = run_experiment(lr_cfg, data)
    save_result(lr_result, f"{OUT_DIR}/results")
    plot_run(lr_result, f"{OUT_DIR}/figures/{exp_id}.png")
    lr_search_results.append(lr_result)
    s = lr_result["summary"]
    print(f"lr={lr:g}: best val macro-F1={s['val_macro_f1']:.4f}, "
          f"best val loss={s['best_val_loss']:.4f}, epoch={s['best_epoch']}, diverged={s['diverged']}")

selected_lr_result = max(lr_search_results, key=lambda r: r["summary"]["val_macro_f1"])
selected_lr = float(selected_lr_result["cfg"]["lr"])
print(f"Learning rate baseline được chọn bằng val: {selected_lr:g}; "
      f"exp_id={selected_lr_result['cfg']['exp_id']}")

# Baseline chính thức: 20 epoch, cùng lr, hai seed; split validation vẫn giữ seed 42.
baseline_cfg = {**DEFAULT_CFG, "lr": selected_lr}
baseline_seeds = [42, 123]
baseline_results = []
for seed in baseline_seeds:
    exp_id = f"{run_tag}-base-s{seed}"
    seed_cfg = {**baseline_cfg, "exp_id": exp_id, "group": "baseline",
                "description": f"Baseline M-base, seed {seed}", "seed": seed}
    seed_result = run_experiment(seed_cfg, data)
    save_result(seed_result, f"{OUT_DIR}/results")
    plot_run(seed_result, f"{OUT_DIR}/figures/{exp_id}.png")
    baseline_results.append(seed_result)
    s = seed_result["summary"]
    print(f"{exp_id}: best_epoch={s['best_epoch']}, best_val_loss={s['best_val_loss']:.4f}, "
          f"val_acc={s['val_acc']:.4f}, val_macro_f1={s['val_macro_f1']:.4f}, "
          f"thời gian/epoch={s['time_per_epoch_s']:.3f}s, diverged={s['diverged']}")

baseline_acc_values = [r["summary"]["val_acc"] for r in baseline_results]
baseline_f1_values = [r["summary"]["val_macro_f1"] for r in baseline_results]
baseline_acc_mean = statistics.mean(baseline_acc_values)
baseline_acc_std = statistics.stdev(baseline_acc_values)
baseline_f1_mean = statistics.mean(baseline_f1_values)
baseline_f1_std = statistics.stdev(baseline_f1_values)
noise_2sigma = 2.0 * baseline_f1_std
baseline_reference = max(baseline_results, key=lambda r: r["summary"]["val_macro_f1"])
baseline_seed_ids = [r["cfg"]["exp_id"] for r in baseline_results]
print(f"Baseline val accuracy = {baseline_acc_mean:.4f} ± {baseline_acc_std:.4f}")
print(f"Baseline val macro-F1 = {baseline_f1_mean:.4f} ± {baseline_f1_std:.4f}; 2σ={noise_2sigma:.4f}")

lr=0.01: best val macro-F1=0.6572, best val loss=0.4478, epoch=5, diverged=False


lr=0.03: best val macro-F1=0.7043, best val loss=0.3762, epoch=5, diverged=False


lr=0.1: best val macro-F1=0.7545, best val loss=0.3296, epoch=5, diverged=False
Learning rate baseline được chọn bằng val: 0.1; exp_id=20261003-122104-lr-sgdm-0p1


20261003-122104-base-s42: best_epoch=18, best_val_loss=0.2305, val_acc=0.9057, val_macro_f1=0.8522, thời gian/epoch=0.484s, diverged=False


20261003-122104-base-s123: best_epoch=19, best_val_loss=0.2231, val_acc=0.9121, val_macro_f1=0.8517, thời gian/epoch=0.475s, diverged=False
Baseline val accuracy = 0.9089 ± 0.0045
Baseline val macro-F1 = 0.8519 ± 0.0004; 2σ=0.0007


**Nhận xét baseline:** Lượt dò 5 epoch cho macro-F1 validation lần lượt là 0,6572 (`lr=0,01`), 0,7043 (`lr=0,03`) và 0,7545 (`lr=0,1`), nên chọn `lr=0,1`. Hai baseline 20 epoch đạt macro-F1 0,8522 và 0,8517, accuracy 0,9057 và 0,9121; trung bình lần lượt là **0,8519 ± 0,0004** và **0,9089 ± 0,0045**. Ngưỡng `2σ` macro-F1 là 0,0007, trong khi chênh lệch hai seed là 0,0005; với chỉ hai seed, chưa có cơ sở nói seed nào tốt hơn. Train loss tiếp tục giảm. Validation loss thấp nhất ở epoch 18–19 rồi nhích lên ở epoch cuối, cho thấy dấu hiệu quá khớp nhẹ ở cuối. Cả hai accuracy đều vượt mốc đoán lớp đa số 0,4876 rõ rệt. Chạy trên CPU nên `peak_mem_MB` để trống (`None`), không suy diễn bộ nhớ GPU.

## Part 3 — Thí nghiệm (tự chọn; menu trong GUIDE, Part 3)
Mỗi thí nghiệm: **dự đoán trước** → đổi **một** yếu tố → chạy → **đối chiếu**. Lặp lại khối dưới cho từng thí nghiệm bạn chọn.
Chủ đề: loss · optimizer · hparam · dropout · clipping · amp · init.

### Dự đoán trước khi chạy Part 3

- **Loss:** MSE trên logits có thể hội tụ chậm hơn CE; sẽ so bằng accuracy, macro-F1 và đường hội tụ, không so trị số loss giữa hai thang đo.
- **Optimizer:** SGD thường nhạy với learning rate hơn Adam. Adam và AdamW với `weight_decay=0` dự kiến gần như trùng nhau. So mỗi optimizer ở hai learning rate trong cùng 20 epoch, seed 42.
- **Batch:** batch 1024 sẽ có ít bước cập nhật hơn batch 512 trong cùng số epoch; có thể giảm thời gian mỗi epoch nhưng học chậm hơn theo epoch.
- **Dropout:** `p=0,2` có thể thu hẹp khoảng cách train–val nếu có quá khớp; baseline chỉ có dấu hiệu nhẹ ở cuối nên không chắc macro-F1 tăng.
- **Clipping:** thử cặp stress test ở learning rate gấp 10 lần baseline. Ngưỡng lấy từ grad norm baseline; clipping được kỳ vọng hạn chế các bước cập nhật lớn.
- **Mixed precision:** CUDA không khả dụng. Thử BF16 autocast trên CPU nếu backend chạy được; dự đoán không nhanh hơn CPU FP32 và không có số đo bộ nhớ GPU.
- **Initialization:** thống kê độ lệch chuẩn sau mỗi Linear. Xavier có thể giữ tín hiệu ổn định; zero-init làm các lớp ẩn đối xứng và ReLU(0)=0 nên gradient tầng ẩn dự kiến bằng 0.

In [5]:
part3_results = []

def run_and_save(exp_cfg):
    result = run_experiment(exp_cfg, data)
    save_result(result, f"{OUT_DIR}/results")
    plot_run(result, f"{OUT_DIR}/figures/{exp_cfg['exp_id']}.png")
    part3_results.append(result)
    s = result["summary"]
    print(f"{exp_cfg['exp_id']}: val_acc={s['val_acc']:.4f}, val_macro_f1={s['val_macro_f1']:.4f}, "
          f"best_epoch={s['best_epoch']}, best_val_loss={s['best_val_loss']:.4f}, "
          f"time/epoch={s['time_per_epoch_s']:.3f}s, peak_mem_MB={s['peak_mem_MB']}, diverged={s['diverged']}")
    return result

baseline_s42 = next(r for r in baseline_results if r["cfg"]["seed"] == 42)

# Chủ đề loss: chỉ đổi CE thành MSE(logits, one-hot), với mean trên batch × 7 lớp.
mse_result = run_and_save({**baseline_cfg, "exp_id": f"{run_tag}-loss-mse-s42", "group": "loss",
                           "description": "MSE giữa logits và one-hot, mean trên mọi phần tử", "loss": "mse", "seed": 42})

# Chủ đề optimizer: mỗi optimizer có hai learning rate; AdamW dùng weight_decay=0 để cô lập optimizer.
optimizer_results = [baseline_s42]
optimizer_lr_grid = {
    "sgd": [0.03, 0.1],
    "sgd_momentum": [0.03, 0.1],
    "adam": [0.001, 0.003],
    "adamw": [0.001, 0.003],
}
for optimizer_name, rates in optimizer_lr_grid.items():
    for lr in rates:
        if optimizer_name == "sgd_momentum" and abs(lr - selected_lr) < 1e-12 and baseline_s42["cfg"]["optimizer"] == optimizer_name:
            print(f"Dùng lại baseline cùng cấu hình cho sgd_momentum lr={lr:g}: {baseline_s42['cfg']['exp_id']}")
            continue
        label = f"{lr:g}".replace(".", "p")
        optimizer_result = run_and_save({**baseline_cfg, "exp_id": f"{run_tag}-opt-{optimizer_name}-{label}-s42",
            "group": "optimizer", "description": f"So optimizer {optimizer_name}, lr={lr:g}",
            "optimizer": optimizer_name, "lr": lr, "weight_decay": 0.0, "seed": 42})
        optimizer_results.append(optimizer_result)

optimizer_best = {}
for optimizer_name in optimizer_lr_grid:
    candidates = [r for r in optimizer_results if r["cfg"]["optimizer"] == optimizer_name]
    optimizer_best[optimizer_name] = max(candidates, key=lambda r: r["summary"]["val_macro_f1"])
    best = optimizer_best[optimizer_name]
    print(f"Optimizer tốt nhất theo val, {optimizer_name}: lr={best['cfg']['lr']:g}, "
          f"macro-F1={best['summary']['val_macro_f1']:.4f}, epoch={best['summary']['best_epoch']}")

# Chủ đề hyperparameter: chỉ tăng batch 512 → 1024, giữ cùng 20 epoch và seed.
batch_result = run_and_save({**baseline_cfg, "exp_id": f"{run_tag}-hparam-batch1024-s42", "group": "hparam",
                             "description": "Batch 1024 thay cho 512", "batch": 1024, "seed": 42})

# Chủ đề dropout: chỉ thêm p=0.2.
dropout_result = run_and_save({**baseline_cfg, "exp_id": f"{run_tag}-dropout-p02-s42", "group": "dropout",
                               "description": "Dropout p=0.2 sau ReLU lớp ẩn", "dropout": 0.2, "seed": 42})

# Chủ đề clipping: stress pair dùng cùng lr cao; c bằng 90% median grad norm trung bình mỗi epoch của baseline.
baseline_norm_median = float(np.median(baseline_s42["history"]["grad_norm"]))
clip_threshold = 0.9 * baseline_norm_median
high_lr = 10.0 * selected_lr
print(f"Ngưỡng clip={clip_threshold:.4f}, lấy từ median grad_norm baseline={baseline_norm_median:.4f}; lr stress={high_lr:g}")
clip_none_result = run_and_save({**baseline_cfg, "exp_id": f"{run_tag}-clip-none-highlr-s42", "group": "clipping",
                                "description": "Stress test lr cao, không clipping", "lr": high_lr,
                                "clip_norm": None, "seed": 42})
clip_on_result = run_and_save({**baseline_cfg, "exp_id": f"{run_tag}-clip-on-highlr-s42", "group": "clipping",
                              "description": f"Stress test cùng lr cao, clip_norm={clip_threshold:.4f}",
                              "lr": high_lr, "clip_norm": clip_threshold, "seed": 42})
clip_results = [clip_none_result, clip_on_result]
clip_fractions = np.asarray(clip_on_result["history"]["clip_fraction"], dtype=float)
print(f"Tỷ lệ batch thực sự bị clipping: {clip_fractions.mean():.3%}; "
      f"theo epoch={[round(float(value), 4) for value in clip_fractions]}")

# Chủ đề mixed precision: CPU BF16 autocast; nếu backend từ chối phép toán thì ghi limitation, không gán metric.
precision_result = None
precision_skip_note = ""
try:
    with torch.autocast(device_type="cpu", dtype=torch.bfloat16):
        _bf16_probe = torch.nn.functional.linear(torch.ones(2, 54), torch.ones(8, 54))
    precision_result = run_and_save({**baseline_cfg, "exp_id": f"{run_tag}-precision-bf16-cpu-s42", "group": "amp",
                                     "description": "BF16 autocast trên CPU; không phải GPU mixed precision",
                                     "precision": "bf16", "seed": 42})
    precision_result["cfg"]["notes"] = "CPU autocast BF16; CUDA không khả dụng; bộ nhớ GPU không áp dụng."
except (RuntimeError, NotImplementedError) as error:
    precision_skip_note = f"Không chạy được CPU BF16 autocast: {type(error).__name__}: {error}"
    print(precision_skip_note)

# Chủ đề init: std được đo sau mỗi Linear (trước ReLU), trên 1.024 mẫu validation ở bước 0.
init_diagnostics = {}
for init_name in ("he", "xavier", "zeros"):
    set_seed(42)
    init_model = MLP(hidden=(256, 128), dropout=0.0, init=init_name).to(device)
    init_metrics = evaluate(init_model, data["X_val"], data["y_val"], loss_name="ce")
    layer_stds = activation_stats(init_model, data["X_val"][:1024])
    init_diagnostics[init_name] = {"step0_loss": init_metrics["loss"], "activation_std_after_linear": layer_stds}
    print(f"init={init_name}: step0 val CE={init_metrics['loss']:.4f}; std sau Linear={np.round(layer_stds, 4).tolist()}")

set_seed(42)
zero_model = MLP(hidden=(256, 128), dropout=0.0, init="zeros").to(device)
zero_model.zero_grad(set_to_none=True)
zero_loss = torch.nn.functional.cross_entropy(zero_model(data["X_tr"][:128]), data["y_tr"][:128])
zero_loss.backward()
zero_gradient_norms = {name: float(param.grad.norm().item()) for name, param in zero_model.named_parameters()}
print("Gradient zero-init sau backward:", zero_gradient_norms)
assert all(zero_gradient_norms[name] == 0.0 for name in ("net.0.weight", "net.0.bias", "net.2.weight", "net.2.bias", "net.4.weight"))
assert zero_gradient_norms["net.4.bias"] > 0.0

xavier_result = run_and_save({**baseline_cfg, "exp_id": f"{run_tag}-init-xavier-s42", "group": "init",
                              "description": "Xavier normal, bias=0", "init": "xavier", "seed": 42})
zero_result = run_and_save({**baseline_cfg, "exp_id": f"{run_tag}-init-zeros-s42", "group": "init",
                            "description": "Zero weights, bias=0", "init": "zeros", "seed": 42})

print(f"Đã chạy {len(part3_results)} experiment Part 3; các lần diverged vẫn được giữ trong results/.")

20261003-122104-loss-mse-s42: val_acc=0.8678, val_macro_f1=0.7374, best_epoch=19, best_val_loss=0.0296, time/epoch=0.465s, peak_mem_MB=None, diverged=False


20261003-122104-opt-sgd-0p03-s42: val_acc=0.8129, val_macro_f1=0.6400, best_epoch=16, best_val_loss=0.4459, time/epoch=0.464s, peak_mem_MB=None, diverged=False


20261003-122104-opt-sgd-0p1-s42: val_acc=0.8530, val_macro_f1=0.7446, best_epoch=18, best_val_loss=0.3552, time/epoch=0.465s, peak_mem_MB=None, diverged=False


20261003-122104-opt-sgd_momentum-0p03-s42: val_acc=0.8851, val_macro_f1=0.8044, best_epoch=16, best_val_loss=0.2789, time/epoch=0.477s, peak_mem_MB=None, diverged=False
Dùng lại baseline cùng cấu hình cho sgd_momentum lr=0.1: 20261003-122104-base-s42


20261003-122104-opt-adam-0p001-s42: val_acc=0.9023, val_macro_f1=0.8486, best_epoch=19, best_val_loss=0.2420, time/epoch=0.523s, peak_mem_MB=None, diverged=False


...

20261003-122104-clip-none-highlr-s42: val_acc=0.8422, val_macro_f1=0.6234, best_epoch=19, best_val_loss=0.4014, time/epoch=0.471s, peak_mem_MB=None, diverged=False


20261003-122104-clip-on-highlr-s42: val_acc=0.8875, val_macro_f1=0.7948, best_epoch=17, best_val_loss=0.2901, time/epoch=0.456s, peak_mem_MB=None, diverged=False


20261003-122104-precision-bf16-cpu-s42: val_acc=0.9094, val_macro_f1=0.8583, best_epoch=20, best_val_loss=0.2270, time/epoch=7.128s, peak_mem_MB=None, diverged=False
init=he: step0 val CE=2.3776; std sau Linear=[0.701, 0.6655, 0.6561]
init=xavier: step0 val CE=2.0429; std sau Linear=[0.2926, 0.2268, 0.2177]
init=zeros: step0 val CE=1.9459; std sau Linear=[0.0, 0.0, 0.0]
Gradient zero-init sau backward: {'net.0.weight': 0.0, 'net.0.bias': 0.0, 'net.2.weight': 0.0, 'net.2.bias': 0.0, 'net.4.weight': 0.0, 'net.4.bias': 0.5005751252174377}


20261003-122104-init-xavier-s42: val_acc=0.9124, val_macro_f1=0.8553, best_epoch=18, best_val_loss=0.2194, time/epoch=0.446s, peak_mem_MB=None, diverged=False


20261003-122104-init-zeros-s42: val_acc=0.4876, val_macro_f1=0.0936, best_epoch=11, best_val_loss=1.2052, time/epoch=0.446s, peak_mem_MB=None, diverged=False
Đã chạy 15 experiment Part 3; các lần diverged vẫn được giữ trong results/.


**Đối chiếu Part 3 (chỉ dùng validation):** Baseline seed 42 đạt macro-F1 0,8522; hai seed cho trung bình 0,8519 ± 0,0004, nên mốc 2σ là 0,0007. Đây chỉ là ước lượng sơ bộ từ hai seed.

- **Loss:** MSE trên logits đạt macro-F1 0,7374 (`20261003-122104-loss-mse-s42`), thấp hơn CE 0,8522. Hai loss có thang đo khác nhau nên đối chiếu bằng macro-F1/accuracy và đường hội tụ, không so trực tiếp trị số loss. Ảnh: `figures/compare_loss.png`.
- **Optimizer:** learning rate tốt nhất trong lưới đã thử: SGD 0,1 → 0,7446; SGD+momentum 0,1 → 0,8522; Adam 0,003 → 0,8712; AdamW 0,003 → 0,8712. Adam tăng 0,0190 so với baseline, lớn hơn mốc 2σ; Adam và AdamW trùng nhau vì `weight_decay=0`. Chỉ thử hai learning rate mỗi optimizer nên kết luận bị giới hạn bởi lưới hẹp. Ảnh: `figures/compare_optimizer.png`.
- **Batch:** tăng 512 → 1024 làm macro-F1 giảm 0,8522 → 0,8335; thời gian/epoch giảm 0,484 → 0,382 giây. Số batch mỗi epoch giảm từ 727 xuống 364, nên cùng 20 epoch không tương đương số bước cập nhật. Ảnh: `figures/compare_hparam.png`.
- **Dropout:** `p=0,2` đạt 0,8161. Khoảng train–val loss cuối giảm từ 0,0240 ở baseline xuống 0,0082, nhưng macro-F1 cũng giảm; dữ liệu này không cho thấy dropout cải thiện chất lượng validation. Ảnh: `figures/compare_dropout.png`.
- **Clipping:** stress test dùng `lr=1` và ngưỡng 0,5065, lấy bằng 90% median grad norm trung bình theo epoch của baseline. Không clip đạt macro-F1 0,6234; có clip đạt 0,7948; cả hai không diverge. Lặp lại đúng seed/cấu hình có macro-F1 khớp 0,7948 và đo được 0,2% batch bị clip (tập trung chủ yếu ở epoch đầu), nên clipping có kích hoạt nhưng hiếm. Không dùng clipping trong cấu hình cuối. Ảnh: `figures/compare_clipping.png`.
- **Mixed precision:** CUDA không khả dụng nên thử BF16 autocast trên CPU. Macro-F1 0,8583 so với FP32 0,8522, nhưng thời gian/epoch 7,128 so với 0,484 giây (khoảng 14,7 lần chậm hơn). Không có số đo bộ nhớ GPU vì chạy CPU. Ảnh: `figures/compare_amp.png`.
- **Khởi tạo:** He có độ lệch chuẩn sau Linear là [0,7010; 0,6655; 0,6561], loss bước 0 là 2,3776; Xavier [0,2926; 0,2268; 0,2177], loss 2,0429; zero [0; 0; 0], loss 1,9459. Zero-init làm gradient các weight/bias tầng ẩn bằng 0 sau backward; chỉ bias output có gradient, phù hợp với đối xứng và `ReLU(0)=0`. Macro-F1 sau huấn luyện lần lượt: He 0,8522, Xavier 0,8553, zero 0,0936. Ảnh: `figures/compare_init.png`.

Adam/AdamW với `lr=0,003` được chọn cho cấu hình cuối theo validation: macro-F1 0,8712, cao hơn baseline 0,0190 và cao nhất trong các cấu hình không diverge. Các chênh lệch nhỏ hơn 2σ cần được xem thận trọng; riêng ngưỡng nhiễu hiện chỉ dựa trên hai seed baseline.


In [6]:
# Ảnh so sánh theo nhóm dùng các lần chạy 20 epoch có cùng seed, trừ lượt dò lr 5 epoch.
plot_compare([baseline_s42, mse_result], "val_macro_f1", f"{OUT_DIR}/figures/compare_loss.png", "CE và MSE: validation macro-F1")
plot_compare(optimizer_results, "val_macro_f1", f"{OUT_DIR}/figures/compare_optimizer.png", "Optimizer ở các learning rate đã thử")
plot_compare([baseline_s42, batch_result], "val_macro_f1", f"{OUT_DIR}/figures/compare_hparam.png", "Batch 512 và 1024")
plot_compare([baseline_s42, dropout_result], "val_macro_f1", f"{OUT_DIR}/figures/compare_dropout.png", "Dropout p=0.2")
plot_compare(clip_results, "val_macro_f1", f"{OUT_DIR}/figures/compare_clipping.png", "Clipping ở learning rate stress")
if precision_result is not None:
    plot_compare([baseline_s42, precision_result], "val_macro_f1", f"{OUT_DIR}/figures/compare_amp.png", "CPU FP32 và CPU BF16 autocast")
plot_compare([baseline_s42, xavier_result, zero_result], "val_macro_f1", f"{OUT_DIR}/figures/compare_init.png", "He, Xavier và zero initialization")

# Khoá cấu hình cuối cùng chỉ bằng validation, trước khi chạy bất cứ đánh giá eval nào.
validation_candidates = baseline_results + [r for r in part3_results if not r["summary"]["diverged"]]
locked_source = max(validation_candidates, key=lambda r: r["summary"]["val_macro_f1"])
locked_cfg = {k: v for k, v in locked_source["cfg"].items() if k not in ("exp_id", "group", "description")}
final_cfg = {**locked_cfg, "exp_id": f"{run_tag}-final", "group": "final",
             "description": f"Cấu hình cuối chọn bằng val từ {locked_source['cfg']['exp_id']}"}
print("FINAL CONFIGURATION LOCKED — chọn theo validation, trước Part 4")
for key in ("hidden", "init", "loss", "optimizer", "lr", "momentum", "betas", "eps", "weight_decay", "batch", "epochs", "dropout", "clip_norm", "precision", "seed"):
    print(f"{key} = {final_cfg.get(key)}")
print(f"Nguồn so sánh val: {locked_source['cfg']['exp_id']}; best_epoch={locked_source['summary']['best_epoch']}; "
      f"val_macro_f1={locked_source['summary']['val_macro_f1']:.4f}; val_loss={locked_source['summary']['best_val_loss']:.4f}")
final_result = run_and_save(final_cfg)

print("Optimizer tốt nhất theo từng nhóm:")
for optimizer_name, result in optimizer_best.items():
    print(f"{optimizer_name}: lr={result['cfg']['lr']:g}, val_macro_f1={result['summary']['val_macro_f1']:.4f}")
if precision_skip_note:
    print("Ghi chú mixed precision:", precision_skip_note)

FINAL CONFIGURATION LOCKED — chọn theo validation, trước Part 4
hidden = (256, 128)
init = he
loss = ce
optimizer = adam
lr = 0.003
momentum = 0.9
betas = (0.9, 0.999)
eps = 1e-08
weight_decay = 0.0
batch = 512
epochs = 20
dropout = 0.0
clip_norm = None
precision = fp32
seed = 42
Nguồn so sánh val: 20261003-122104-opt-adam-0p003-s42; best_epoch=20; val_macro_f1=0.8712; val_loss=0.2116


20261003-122104-final: val_acc=0.9161, val_macro_f1=0.8712, best_epoch=20, best_val_loss=0.2116, time/epoch=0.493s, peak_mem_MB=None, diverged=False
Optimizer tốt nhất theo từng nhóm:
sgd: lr=0.1, val_macro_f1=0.7446
sgd_momentum: lr=0.1, val_macro_f1=0.8522
adam: lr=0.003, val_macro_f1=0.8712
adamw: lr=0.003, val_macro_f1=0.8712


## Part 4 — Đánh giá cuối trên eval, bảng và báo cáo
Cấu hình cuối đã được khóa ở ô Part 3 **chỉ bằng validation**, trước khi sử dụng eval. Ở đây evaluator chính thức được chạy cho baseline seed 42 và cấu hình cuối; chỉ kết quả cấu hình cuối được lưu vào `predictions_eval.csv` và `eval_result.json`.


In [7]:
# TODO: chọn cấu hình cuối cùng theo VAL (có thể là baseline nếu bạn không kết hợp thêm gì)
# TODO: final_eval(cfg_final, result_final, data, f"{OUT_DIR}/predictions_eval.csv")
# TODO: chạy: python scripts/evaluate.py --pred <OUT_DIR>/predictions_eval.csv --out <OUT_DIR>/eval_result.json (từ REPO_ROOT)
# TODO: đọc eval_result.json: ghi accuracy, macro_f1 vào bảng và báo cáo; làm tương tự cho baseline (predictions riêng, không nộp nếu không cần)

In [8]:
# TODO: phân tích lỗi trên eval: bảng precision/recall/F1 theo lớp (từ eval_result.json) và ma trận nhầm lẫn;
#       lớp nào khó nhất? vì sao (số mẫu ít? nhầm với lớp nào?)  -> viết nhận xét ở ô markdown bên dưới

**Phân tích lỗi (tự viết):** ...

In [9]:
# TODO: results = load_results(f"{OUT_DIR}/results"); rows = [to_row(r, ...) for r in results]
# TODO: write_xlsx(rows, f"{REPO_ROOT}/templates/experiment_table_template.xlsx", f"{OUT_DIR}/experiments.xlsx")
# TODO: bổ sung sheet Seeds (exp_id các lần chạy baseline) và nhận xét ở sheet Summary